In [1]:
import pandas as pd
import os

EXPORT_PATH = r"C:\Users\yipch\OneDrive\Desktop\olist_analytics\exports"

master       = pd.read_csv(os.path.join(EXPORT_PATH, "master_orders.csv"),
                           parse_dates=['order_purchase_timestamp',
                                        'order_delivered_customer_date',
                                        'order_estimated_delivery_date'])
rfm          = pd.read_csv(os.path.join(EXPORT_PATH, "rfm_segments.csv"))
ltv          = pd.read_csv(os.path.join(EXPORT_PATH, "customer_ltv.csv"))
monthly      = pd.read_csv(os.path.join(EXPORT_PATH, "monthly_trends.csv"))
pareto       = pd.read_csv(os.path.join(EXPORT_PATH, "pareto_category.csv"))
delivery     = pd.read_csv(os.path.join(EXPORT_PATH, "delivery_by_state.csv"))
payments     = pd.read_csv(os.path.join(EXPORT_PATH, "payment_behavior.csv"))
kpi          = pd.read_csv(os.path.join(EXPORT_PATH, "kpi_summary.csv"))
freq         = pd.read_csv(os.path.join(EXPORT_PATH, "order_frequency.csv"))
dow          = pd.read_csv(os.path.join(EXPORT_PATH, "dow_trends.csv"))

print("✅ All exports loaded!")

✅ All exports loaded!


In [2]:
# Merge RFM segment into master
master_final = master.merge(
    rfm[['customer_unique_id','R_score','F_score','M_score','segment']],
    on='customer_unique_id', how='left'
)

# Merge LTV segment into master
master_final = master_final.merge(
    ltv[['customer_unique_id','ltv_segment','total_orders','avg_order_value']],
    on='customer_unique_id', how='left'
)

# Clean column names (Power BI friendly — no spaces)
master_final.columns = master_final.columns.str.strip().str.lower().str.replace(' ','_')

print(f"✅ Final master shape: {master_final.shape}")
print(master_final.dtypes)

✅ Final master shape: (96470, 30)
order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                           str
order_delivered_carrier_date                str
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
order_year_month                            str
order_year                                int64
order_month                               int64
order_dow                                   str
delivery_days                             int64
delivery_delay_days                       int64
customer_unique_id                          str
customer_city                               str
customer_state                              str
total_items                               int64
total_revenue                           float64
total_freight                           float64
total_

In [3]:
# Power BI reads these formats cleanly
master_final['order_purchase_timestamp'] = (
    master_final['order_purchase_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')
)
master_final['order_delivered_customer_date'] = (
    master_final['order_delivered_customer_date'].dt.strftime('%Y-%m-%d %H:%M:%S')
)
master_final['order_estimated_delivery_date'] = (
    master_final['order_estimated_delivery_date'].dt.strftime('%Y-%m-%d %H:%M:%S')
)

print("✅ Dates formatted for Power BI!")

✅ Dates formatted for Power BI!


In [4]:
files = {
    "final_master.csv":        master_final,
    "final_rfm.csv":           rfm,
    "final_ltv.csv":           ltv,
    "final_monthly.csv":       monthly,
    "final_pareto.csv":        pareto,
    "final_delivery.csv":      delivery,
    "final_payments.csv":      payments,
    "final_kpi.csv":           kpi,
    "final_order_frequency.csv": freq,
    "final_dow.csv":           dow,
}

for filename, df in files.items():
    df.to_csv(os.path.join(EXPORT_PATH, filename), index=False)
    print(f"  ✅ Saved: {filename}  ({len(df):,} rows)")

print("\n🎉 All final files saved to exports/!")

  ✅ Saved: final_master.csv  (96,470 rows)
  ✅ Saved: final_rfm.csv  (93,350 rows)
  ✅ Saved: final_ltv.csv  (93,350 rows)
  ✅ Saved: final_monthly.csv  (21 rows)
  ✅ Saved: final_pareto.csv  (72 rows)
  ✅ Saved: final_delivery.csv  (27 rows)
  ✅ Saved: final_payments.csv  (5 rows)
  ✅ Saved: final_kpi.csv  (1 rows)
  ✅ Saved: final_order_frequency.csv  (9 rows)
  ✅ Saved: final_dow.csv  (7 rows)

🎉 All final files saved to exports/!


In [5]:
all_files = os.listdir(EXPORT_PATH)
csv_files = [f for f in all_files if f.endswith('.csv')]

print(f"📁 Total CSV files in exports/: {len(csv_files)}")
print()
for f in sorted(csv_files):
    path = os.path.join(EXPORT_PATH, f)
    size = os.path.getsize(path) / 1024
    print(f"  {f:<45} {size:>8.1f} KB")

📁 Total CSV files in exports/: 26

  cohort_matrix.csv                                  1.5 KB
  customer_ltv.csv                                9343.3 KB
  delivery_by_state.csv                              0.9 KB
  delivery_performance.csv                        5170.3 KB
  dow_trends.csv                                     0.2 KB
  final_delivery.csv                                 0.9 KB
  final_dow.csv                                      0.2 KB
  final_kpi.csv                                      0.2 KB
  final_ltv.csv                                   9333.2 KB
  final_master.csv                               30331.4 KB
  final_monthly.csv                                  1.0 KB
  final_order_frequency.csv                          0.1 KB
  final_pareto.csv                                   5.2 KB
  final_payments.csv                                 0.3 KB
  final_rfm.csv                                   6202.5 KB
  installments_dist.csv                              0.2 KB
  kpi

In [8]:
from sqlalchemy import create_engine

# ⚠️ Replace with your MySQL credentials
USER     = "root"
PASSWORD = "829131"
HOST     = "localhost"
PORT     = "3306"
DATABASE = "olist_db"

engine = create_engine(f"mysql+mysqlconnector://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}")

tables = {
    "final_master":    master_final,
    "final_rfm":       rfm,
    "final_ltv":       ltv,
    "final_monthly":   monthly,
    "final_pareto":    pareto,
    "final_delivery":  delivery,
    "final_payments":  payments,
    "final_kpi":       kpi,
}

for table_name, df in tables.items():
    df.to_sql(table_name, con=engine, if_exists='replace', index=False)
    print(f"  ✅ Loaded: {table_name}")

print("\n🎉 All tables loaded into MySQL!")

  ✅ Loaded: final_master
  ✅ Loaded: final_rfm
  ✅ Loaded: final_ltv
  ✅ Loaded: final_monthly
  ✅ Loaded: final_pareto
  ✅ Loaded: final_delivery
  ✅ Loaded: final_payments
  ✅ Loaded: final_kpi

🎉 All tables loaded into MySQL!
